# Standardised Sentinel-1 loading from file, API, and geoparquet

For this project, we may have a number of different data sources.
The goal of the `io` module is to standardise the load as much as possible before entering the norm-prod part of the workflow.

## Features
* The loading code will convert linear to decibels with appropriate clipping to avoid introducing -Inf values. To do this, the user must supply the unit of the data (either `linear` or `db`).
* The module provides one function for loading from file, and two functions for working with STAC (one to query, one to load). The STAC functions work for both STAC-API endpoints and local geoparquet files

## Set up
### Imports

In [ ]:
from fast_ice.io import load_sentinel_1_from_file, query_sentinel_1_stac,load_sentinel_1_from_stac, SAR_UNIT
from datetime import datetime, timedelta
import odc.geo.xr
from odc.stac import configure_s3_access
import pandas as pd

import os
os.environ["AWS_DEFAULT_REGION"] = "ap-southeast-2"
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

### Download required files for notebook

We provide an cropped EW GeoTIFF to demonstrate how to load from file. 
The GeoTIFF's [GeoBox](https://odc-geo.readthedocs.io/en/latest/intro-geobox.html) is then used to query the STAC API and the STAC GeoParquet file.

The sample file is downloaded to your cache using the `fetch_ew_20191217()` function.
This function will return a `SampleScene` class with the metadata needed to run the example.

In [ ]:
from fast_ice.datasets.sample_data import fetch_ew_20191217

scene = fetch_ew_20191217()

Run the next cell to display the scene path and its associated metadata. 
Usually, you would need to supply this metadata yourself when working from files.

In [ ]:
scene

## Loading from GeoTIFF

The user must supply the band name, band unit, and datetime of the file.
These properties are added as metadata to the loaded xarray.

In [ ]:
# Extract metadata from SampleScene
# Typically, the user would supply this data directly
file_path = scene.path
file_timestamp = scene.timestamp
file_band = scene.band
file_band_unit = scene.band_unit

obs_from_file = load_sentinel_1_from_file(
    file_path, 
    band=file_band, 
    band_unit=file_band_unit, 
    timestamp=file_timestamp
)

obs_from_file

The geobox and timestamp of the file-based xarray are saved for use in the STAC query and loading sections.

In [ ]:
# Save properties to use with stac loading
obs_geobox = obs_from_file.odc.geobox
obs_time = pd.Timestamp(obs_from_file.time.values).to_pydatetime()
obs_search_geometry = obs_geobox.extent.to_crs("EPSG:4326")

obs_geobox

## Load from STAC

### API
When loading IW data to compare, widen the search window to 6 days either side of the EW observation. 
This is only a test to see if we can return an observation -- the choice of 6 days was arbitrary.

#### Get STAC items

In [ ]:
stac_endpoint = "https://explorer.dev.dea.ga.gov.au/stac"
search_collection = "ga_s1_nrb_iw_hh_1"
search_band = "hh_gamma0"
search_unit: SAR_UNIT = "linear"

search_start = obs_time - timedelta(days=6)
search_end = obs_time + timedelta(days=6)

items_from_api = query_sentinel_1_stac(
    search_collection, 
    search_start, 
    search_end, 
    intersects=obs_search_geometry, 
    method="api", 
    api_endpoint=stac_endpoint
)

items_from_api

#### Load STAC items

In [ ]:
obs_from_api = load_sentinel_1_from_stac(
    items_from_api, 
    search_band,
    search_unit, 
    geobox=obs_geobox, 
    groupby="sat:relative_orbit"
)

obs_from_api

### Geoparquet

#### Get STAC items

In [ ]:
geoparquet_file = "data/test/ga_s1_nrb_ew_hh_hv_1_v2.parquet"
search_collection = "ga_s1_nrb_ew_hh_hv_1"
search_band = "hh_gamma0"
search_unit: SAR_UNIT = "linear"

search_start = obs_time - timedelta(hours=1)
search_end = obs_time + timedelta(hours=1)

items_from_geoparquet = query_sentinel_1_stac(
    search_collection, 
    search_start, 
    search_end, 
    intersects=obs_search_geometry, 
    method="geoparquet", 
    geoparquet_file=geoparquet_file
)

items_from_geoparquet

#### Load STAC items

In [ ]:
obs_from_geoparquet = load_sentinel_1_from_stac(
    items_from_geoparquet, 
    search_band,
    search_unit, 
    geobox=obs_geobox, 
    groupby="sat:relative_orbit"
)

obs_from_geoparquet

## Visually compare

In [ ]:
import matplotlib.pyplot as plt

arrays = {
    "EW 2019-12-17 (File)": obs_from_file, 
    "EW 2019-12-17 (Geoparquet)": obs_from_geoparquet, 
    "IW 2019-12-19 (API)": obs_from_api
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (label, da) in zip(axes, arrays.items()):
    da.plot(ax=ax, vmin=-25, vmax=5, add_colorbar=False)
    ax.set_title(label)

plt.tight_layout()
plt.show()